In [ ]:
"""
LLM 실습 과제 정답지
- API Key가 없을 경우 문제 7,8은 자동으로 skip
"""

# =========================
# 문제 1
# =========================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "LLM is powerful"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("[정답1]", tokens, token_ids, decoded_tokens)


# =========================
# 문제 2
# =========================
import torch
import torch.nn.functional as F

Q = torch.randn(2,4)
K = torch.randn(2,4)
V = torch.randn(2,4)

d = Q.size(-1)

scores = torch.matmul(Q, K.t()) / (d ** 0.5)
weights = F.softmax(scores, dim=-1)
output = torch.matmul(weights, V)

print("[정답2]", output)


# =========================
# 문제 3
# =========================
import time


def attention_compute(x):
    return torch.matmul(x, x.T)


def run_no_cache(seq_len=50):
    x = torch.randn(seq_len, 64)
    start = time.time()
    for i in range(1, seq_len+1):
        _ = attention_compute(x[:i])
    return time.time() - start


def run_with_cache(seq_len=50):
    cache = []
    start = time.time()
    for i in range(1, seq_len+1):
        new_token = torch.randn(1, 64)
        cache.append(new_token)
        x = torch.cat(cache, dim=0)
        _ = attention_compute(x[-1:].repeat(x.shape[0],1))
    return time.time() - start, len(cache)


t1 = run_no_cache()
t2, cache_size = run_with_cache()

print("[정답3]", t1, t2, cache_size)


# =========================
# 문제 4
# =========================
messages = [
    {"role": "system", "content": "You are helpful"},
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi there"}
]

print("[정답4]", messages)


# =========================
# 문제 5
# =========================
import json


def get_weather(city): return f"{city} sunny"

args = json.loads('{"city":"Seoul"}')
result = get_weather(**args)

print("[정답5]", result)


# =========================
# 문제 6
# =========================

def add(a,b): return a+b

def multiply(a,b): return a*b


def tool_router(name,args):
    if name == "add":
        return add(**args)
    elif name == "multiply":
        return multiply(**args)

print("[정답6]", tool_router("multiply", {"a":3,"b":4}))


# =========================
# 문제 7 (API 필요)
# =========================
try:
    from openai import OpenAI
    client = OpenAI(api_key="YOUR_API_KEY")

    def calculator(a,b): return a+b

    messages = [{"role": "user", "content": "10 + 20 계산해줘"}]

    tools = [{
        "type":"function",
        "function":{
            "name":"calculator",
            "parameters":{
                "type":"object",
                "properties":{
                    "a":{"type":"number"},
                    "b":{"type":"number"}
                },
                "required":["a","b"]
            }
        }
    }]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message
    messages.append(msg)

    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    result = calculator(**args)

    messages.append({
        "role": "tool",
        "content": str(result),
        "tool_call_id": tool_call.id
    })

    final = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    print("[정답7]", final.choices[0].message.content)
except Exception as e:
    print("[정답7] skipped (API 필요)")


# =========================
# 문제 8 (API 필요)
# =========================
try:
    cache = []
    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": "Explain AI"}],
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        if hasattr(delta, "content") and delta.content:
            cache.append(delta.content)
            if len("".join(cache)) > 100:
                break

    print("[정답8]", "".join(cache))
except Exception:
    print("[정답8] skipped (API 필요)")


# =========================
# 문제 9
# =========================
import requests

BASE_URL = "https://open.er-api.com/v6/latest/"


def get_exchange_rate(base, target):
    try:
        data = requests.get(BASE_URL + base).json()
        return data["rates"][target]
    except:
        return "Error"

print("[정답9]", get_exchange_rate("USD","KRW"), get_exchange_rate("USD","INVALID"))


# =========================
# 문제 10
# =========================
expr = "3 더하기 5 곱하기 2"
expr = expr.replace("더하기", "+").replace("곱하기", "*")
parts = expr.split()

# parsing
tokens = []
for p in parts:
    if p in ["+","*"]:
        tokens.append(p)
    else:
        tokens.append(int(p))

# evaluation
def compute(a, op, b):
    if op == "+": return a + b
    if op == "*": return a * b

res = tokens[0]
i = 1
while i < len(tokens):
    res = compute(res, tokens[i], tokens[i+1])
    i += 2

print("[정답10]", res)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[정답1] ['ll', '##m', 'is', 'powerful'] [2222, 2213, 2003, 3928] ['ll', '##m', 'is', 'powerful']
[정답2] tensor([[ 0.3284, -0.5114, -1.4551,  0.8785],
        [ 0.1618, -0.5366, -1.3461,  0.8618]])
[정답3] 0.014341115951538086 0.01791977882385254 50
[정답4] [{'role': 'system', 'content': 'You are helpful'}, {'role': 'user', 'content': 'Hello'}, {'role': 'assistant', 'content': 'Hi there'}]
[정답5] Seoul sunny
[정답6] 12
[정답7] 10 + 20 = 30입니다.
[정답8] AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed
[정답9] 1482.511805 Error
[정답10] 16
